In [1]:
# Project Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

warnings.filterwarnings("ignore")

RANDOM_STATE = 121

In [2]:
# Load Dataset
df = pd.read_csv("../data/raw/raw_donor_data.csv")
raw_df = df.copy()

In [3]:
# Inspect Data: shape, columns, datatypes, memory usage, and missing values
# Dataset Shape
print("Dataset Shape: ")
print(df.shape)

# Column Names
print("\nColumns: ")
print(df.columns.tolist())

# Datatypes
print("\nDatatypes: ")
print(df.dtypes)

# Memory Usage
print("\nTotal Memory Usage: " + f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Missing values
missing_counts = df.isnull().sum()
print("\nColumns Missing Values:")
print(missing_counts[missing_counts > 0].sort_values(ascending=False))

Dataset Shape: 
(34508, 23)

Columns: 
['DonorUniqueId', 'DonorPostalCode', 'DonorAge', 'MaritalStatus', 'GenderIdentity', 'IsMemberFlag', 'IsAlumnusFlag', 'IsParentFlag', 'HasInvolvementFlag', 'WealthRating', 'AcademicDegreeLevel', 'PreferredAddressType', 'HasEmailFlag', 'ConsecutiveDonorYears', 'LastFiscalYearDonation', 'Donation2FiscalYearsAgo', 'Donation3FiscalYearsAgo', 'Donation4FiscalYearsAgo', 'Donation5FiscalYearsAgo', 'CurrentFiscalYearDonation', 'CumulativeDonationAmount', 'DonorDateOfBirth', 'DonorIndicatorFlag.']

Datatypes: 
DonorUniqueId                  int64
DonorPostalCode              float64
DonorAge                       int64
MaritalStatus                    str
GenderIdentity                   str
IsMemberFlag                     str
IsAlumnusFlag                    str
IsParentFlag                     str
HasInvolvementFlag               str
WealthRating                     str
AcademicDegreeLevel              str
PreferredAddressType             str
HasEmailFla

## Initial Dataset Characteristics
- The dataset contains 34,508 rows and 23 columns
- The dataset size is around 7.2 MB
  - The size is relatively manageable
  - This allows full in-memory processing using pandas
- Multiple fields that are expected to be numeric are currently being stored as strings
  - The fields include: 
    - LastFiscalYearDonation
    - Donation2FiscalYearsAgo
    - Donation3FiscalYearsAgo
    - Donation4FiscalYearsAgo
    - Donation5FiscalYearsAgo
    - CurrentFiscalYearDonation
  - These columns will likely require currency symbol removal, comma removal, conversion to numeric datatype
  - Another field that is stored as a string is DonorDateOfBirth which will likely require datetime conversion and validation
  - Several columns that are stored as stings also represent boolean values which may require standardization with binary encoding
    - These fields include:
      - IsMemberFlag
      - IsAlumnusFlag
      - IsParentFlag
      - HasInvolvementFlag
      - HasEmailFlag
      - DonorIndicatorFlag

- Donation-related features span across multiple fiscal years

- Several categorical and demographic columns contain substantial missing values
  - WealthRating is missing 31,799 values
    - This indicates WealthRating is a sparsely populated feature and may require removal or special handling during feature engineering
  - AcademicDegreeLevel is missing 26,902 values
    - The amount of missing values is significant because it may also require removal or special handling during feature engineering
  - MaritalStatus is missing 24,568 values
    - This amount is significant because it indicates most demographic data is incomplete
  - DonorDateOfBirth is missing 21,190 values
    - This demonstrates the limitations with age and date validation

In [4]:
# Sample Rows
df.sample(10, random_state=RANDOM_STATE)

,DonorUniqueId,DonorPostalCode,DonorAge,MaritalStatus,GenderIdentity,IsMemberFlag,IsAlumnusFlag,IsParentFlag,HasInvolvementFlag,WealthRating,AcademicDegreeLevel,PreferredAddressType,HasEmailFlag,ConsecutiveDonorYears,LastFiscalYearDonation,Donation2FiscalYearsAgo,Donation3FiscalYearsAgo,Donation4FiscalYearsAgo,Donation5FiscalYearsAgo,CurrentFiscalYearDonation,CumulativeDonationAmount,DonorDateOfBirth,DonorIndicatorFlag.
16199,16200,90265.0,45,NaN,Male,N,Y,N,N,NaN,GM,HOME,N,0,$0,$0,$0,$0,$0,$0,0.0,8/16/1973,N
25925,25926,45864.0,26,NaN,Uknown,N,Y,N,Y,NaN,UB,BUSN,Y,1,$0,$0,$0,$0,$0,$0,0.0,1/3/1992,N
12681,12682,90265.0,46,NaN,Female,N,Y,N,N,NaN,NaN,HOME,Y,0,$0,$0,$0,$0,$0,$0,0.0,3/19/1972,N
34020,34021,90265.0,42,NaN,Male,N,N,N,N,NaN,NaN,NaN,N,1,$0,$0,$0,$0,$0,$0,25.0,NaN,Y
32576,32577,90265.0,60,NaN,Female,N,N,N,N,NaN,NaN,HOME,N,0,$0,$100,$100,$0,$0,$0,200.0,12/20/1958,Y
9047,9048,37172.0,42,Married,Female,N,N,N,N,NaN,NaN,HOME,N,0,$0,$0,$0,$0,$0,$200,200.0,NaN,Y
2988,2989,8026.0,42,Single,Uknown,N,N,N,Y,NaN,NaN,HOME,N,0,$0,$0,$0,$0,$150,$0,150.0,NaN,Y
31189,31190,27568.0,64,NaN,Male,N,N,N,N,NaN,NaN,HOME,N,0,$0,$0,$0,$0,$0,$0,0.0,6/15/1954,N
2661,2662,15482.0,56,NaN,Male,N,N,N,N,NaN,NaN,HOME,N,0,$25,$0,$0,$0,$0,$0,25.0,3/17/1962,Y
20668,20669,56366.0,28,Married,Female,N,N,Y,N,NaN,NaN,HOME,Y,0,$0,$0,$0,$0,$0,$0,0.0,9/20/1990,N


In [5]:
# Summary table about datatypes, missing values, and column uniqueness

summary_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.values,
    "missing_count": df.isnull().sum().values,
    "missing_pct": (
        df.isnull().sum().values / len(df) * 100
    ),
    "unique_values": [
        df[col].nunique()
        for col in df.columns
    ]
})

summary_df

,column,dtype,missing_count,missing_pct,unique_values
0,DonorUniqueId,int64,0,0.000000,34508
1,DonorPostalCode,float64,91,0.263707,20991
2,DonorAge,int64,0,0.000000,102
3,MaritalStatus,str,24568,71.195085,6
4,GenderIdentity,str,493,1.428654,5
5,IsMemberFlag,str,0,0.000000,1
6,IsAlumnusFlag,str,0,0.000000,2
7,IsParentFlag,str,0,0.000000,2
8,HasInvolvementFlag,str,0,0.000000,2
9,WealthRating,str,31799,92.149646,8


## Summary Table Observations

- DonorUniqueId contains 34,508 unique values, matching the total number of records in the dataset. This confirms that DonorUniqueId is a primary key and will be treated as a unique identifier rather than a predictive feature.
- DonorPostalCode has 20,991 unique values which makes sense as it provides geographic informaiton and will require feature engineering, such as grouping by prefix or target encoding, before modeling.
- Flag variables that represent binary indicators contains no more than 2 unique values. This is consistent with the expected binary classification and will require verification for consistency and determine whether binary encoding is necessary.
  - These columns are: IsMemberFlag, IsAlumnusFlag, IsParentFlag, HasInvolvementFlag, HasEmailFlag, DonorIndicatorFlag
- IsMemberFlag only contains only a single unique value despite having no missing values suggesting that the feature may have zero variance and may provide no predictive value. 
- WealthRating, AcademicDegreeLevel, MaritalStatus, and DonorDateOfBirth exhibit extremely high missing value percentages at 92.15%, 77.96%, 71.2%, and 61.41%, respectively. 

In [6]:
# Categorical Feature Columns
for col in df.columns:
    if df[col].nunique() < 10:
        print(f"\n{'='*50}")
        print(f"Column: {col}")
        print(f"{'='*50}")
        print(df[col].value_counts(dropna=False))


Column: MaritalStatus
MaritalStatus
NaN              24568
Married           6547
Single            3140
Widowed            157
Divorced            84
Separated            9
Never Married        3
Name: count, dtype: int64

Column: GenderIdentity
GenderIdentity
Female     16678
Male       16233
Uknown      1091
NaN          493
Unknown       12
U              1
Name: count, dtype: int64

Column: IsMemberFlag
IsMemberFlag
N    34508
Name: count, dtype: int64

Column: IsAlumnusFlag
IsAlumnusFlag
N    26086
Y     8422
Name: count, dtype: int64

Column: IsParentFlag
IsParentFlag
N    31807
Y     2701
Name: count, dtype: int64

Column: HasInvolvementFlag
HasInvolvementFlag
N    26533
Y     7975
Name: count, dtype: int64

Column: WealthRating
WealthRating
NaN                      31799
$50,000-$99,999            645
$1-$24,999                 580
$25,000-$49,999            564
$100,000-$249,999          511
$250,000-$499,999          265
$500,000-$999,999           81
$1,000,000-$2,499,999 

## Categorical Feature Observations

- MartialStatus contains a significant amount of missing values with around 71% of records missing a value
  - Among known values, 6,547 donors are married and 3,140 donors are single
- GenderIdentity contains inconsistent category labels
  - These labels are: Female, Male, Uknown, Unknown, and U
  - Uknown, Unknown, and U likely represent the same category; further evaluation is required to determine if these values should be standardize into a single category
- IsMemberFlag only contains a single value: N
  - This means the feature has zero variance
  - The column has no ability to distinguish between donors which suggests:
    - There is a data quality issue
    - A population selection rule
    - Or it is a feature with no predicitve value
  - This likely means that the column will be removed during preprocessing
- AcademicDegreeLevel contains coded values: UB, GM, GP, GD, GC, UC, UG
  - The meaning of these codes are currently unknown. Additional documentation or data dictionary review is required before preprocessing
- PreferredAddressType also contains coded values: HOME, BUSN, CAMP, OTR
  - Likely Semantic Meaning
  - HOME = Home Address
  - BUSN = Business Address
  - CAMP = Campus Address
  - OTR = Other
  

In [7]:
# Stat Summary
df.describe(include="all")

,DonorUniqueId,DonorPostalCode,DonorAge,MaritalStatus,GenderIdentity,IsMemberFlag,IsAlumnusFlag,IsParentFlag,HasInvolvementFlag,WealthRating,AcademicDegreeLevel,PreferredAddressType,HasEmailFlag,ConsecutiveDonorYears,LastFiscalYearDonation,Donation2FiscalYearsAgo,Donation3FiscalYearsAgo,Donation4FiscalYearsAgo,Donation5FiscalYearsAgo,CurrentFiscalYearDonation,CumulativeDonationAmount,DonorDateOfBirth,DonorIndicatorFlag.
count,34508.000000,34417.000000,34508.000000,9940,34015,34508,34508,34508,34508,2709,7606,30465,34508,34508.000000,34508,34508,34508,34508,34508,34508,3.450800e+04,13318,34508
unique,NaN,NaN,NaN,6,5,1,2,2,2,8,7,4,2,NaN,188,189,188,177,182,166,NaN,4812,2
top,NaN,NaN,NaN,Married,Female,N,N,N,N,"$50,000-$99,999",UB,HOME,N,NaN,$0,$0,$0,$0,$0,$0,NaN,9/10/1991,Y
freq,NaN,NaN,NaN,6547,16678,34508,26086,31807,26533,645,3648,28771,23221,NaN,32139,32032,31971,32362,32544,32599,NaN,20,21433
mean,17254.500000,56161.625970,43.366987,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.137475,NaN,NaN,NaN,NaN,NaN,NaN,2.363534e+03,NaN,NaN
std,9961.745881,29966.704506,11.405722,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.423034,NaN,NaN,NaN,NaN,NaN,NaN,1.138014e+05,NaN,NaN
min,1.000000,211.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,NaN,NaN
25%,8627.750000,29927.000000,42.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,NaN,NaN
50%,17254.500000,58108.000000,42.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,2.500000e+01,NaN,NaN
75%,25881.250000,90265.000000,42.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,1.440000e+02,NaN,NaN


In [8]:
# dataframe
validation_df = pd.DataFrame({
    "Column": df.columns,
    "Current_Dtype": df.dtypes.values,
    "Missing_Pct": (
        df.isnull().sum() / len(df) * 100
    ).round(2),
    "Unique_Values": [
        df[col].nunique()
        for col in df.columns
    ]
})

validation_df

,Column,Current_Dtype,Missing_Pct,Unique_Values
DonorUniqueId,DonorUniqueId,int64,0.00,34508
DonorPostalCode,DonorPostalCode,float64,0.26,20991
DonorAge,DonorAge,int64,0.00,102
MaritalStatus,MaritalStatus,str,71.20,6
GenderIdentity,GenderIdentity,str,1.43,5
IsMemberFlag,IsMemberFlag,str,0.00,1
IsAlumnusFlag,IsAlumnusFlag,str,0.00,2
IsParentFlag,IsParentFlag,str,0.00,2
HasInvolvementFlag,HasInvolvementFlag,str,0.00,2
WealthRating,WealthRating,str,92.15,8


## Data Dictionary Validation Observations

- Donational Amount Fields are currently stored as strings despite representing monetary values
- Date Fields are also currently stored as strings instead of a datetime datatype
- Binary Indicator Fields are stored as strings but repsent boolean/binary values
- CumulativeDonationAmount appears to be a continuous monetary feature that may require outlier analysis
- DonorPostalCode contains a large number of unique values and may require grouping or feature engineering
- DonorIndicatorFlag. contains a trailing period which should be standardized during preprocessing

In [9]:
# Identify all string columns
object_cols = df.select_dtypes(include=["object"]).columns.tolist()
object_cols

['MaritalStatus',
 'GenderIdentity',
 'IsMemberFlag',
 'IsAlumnusFlag',
 'IsParentFlag',
 'HasInvolvementFlag',
 'WealthRating',
 'AcademicDegreeLevel',
 'PreferredAddressType',
 'HasEmailFlag',
 'LastFiscalYearDonation',
 'Donation2FiscalYearsAgo',
 'Donation3FiscalYearsAgo',
 'Donation4FiscalYearsAgo',
 'Donation5FiscalYearsAgo',
 'CurrentFiscalYearDonation',
 'DonorDateOfBirth',
 'DonorIndicatorFlag.']

In [10]:
# Inspect Potential Missing Value Representations
for col in df.columns:
    if df[col].dtype == "object" or str(df[col].dtype) == "str":
        print(f"\n{'='*50}")
        print(f"Column: {col}")
        print(f"{'='*50}")
        print(df[col].value_counts(dropna=False).head(20))


Column: MaritalStatus
MaritalStatus
NaN              24568
Married           6547
Single            3140
Widowed            157
Divorced            84
Separated            9
Never Married        3
Name: count, dtype: int64

Column: GenderIdentity
GenderIdentity
Female     16678
Male       16233
Uknown      1091
NaN          493
Unknown       12
U              1
Name: count, dtype: int64

Column: IsMemberFlag
IsMemberFlag
N    34508
Name: count, dtype: int64

Column: IsAlumnusFlag
IsAlumnusFlag
N    26086
Y     8422
Name: count, dtype: int64

Column: IsParentFlag
IsParentFlag
N    31807
Y     2701
Name: count, dtype: int64

Column: HasInvolvementFlag
HasInvolvementFlag
N    26533
Y     7975
Name: count, dtype: int64

Column: WealthRating
WealthRating
NaN                      31799
$50,000-$99,999            645
$1-$24,999                 580
$25,000-$49,999            564
$100,000-$249,999          511
$250,000-$499,999          265
$500,000-$999,999           81
$1,000,000-$2,499,999 

In [11]:
# Convert missing placeholder values and whitespace
missing_values = [
    "NA",
    "N/A",
    "NULL",
    "null",
    "None",
    "none",
    "?"
    ""
    " "
]

df.replace(missing_values, np.nan, inplace=True)
df.replace(r"^\s*$", np.nan, regex=True, inplace=True)

,DonorUniqueId,DonorPostalCode,DonorAge,MaritalStatus,GenderIdentity,IsMemberFlag,IsAlumnusFlag,IsParentFlag,HasInvolvementFlag,WealthRating,AcademicDegreeLevel,PreferredAddressType,HasEmailFlag,ConsecutiveDonorYears,LastFiscalYearDonation,Donation2FiscalYearsAgo,Donation3FiscalYearsAgo,Donation4FiscalYearsAgo,Donation5FiscalYearsAgo,CurrentFiscalYearDonation,CumulativeDonationAmount,DonorDateOfBirth,DonorIndicatorFlag.
0,1,23187.0,42,Married,Female,N,N,N,N,NaN,NaN,HOME,N,1,$0,$0,$0,$0,$0,$0,10.0,NaN,Y
1,2,77643.0,33,NaN,Female,N,Y,N,Y,NaN,UB,NaN,Y,0,$0,$0,$0,$0,$0,$0,2100.0,6/16/1985,Y
2,3,NaN,42,Married,Female,N,N,N,N,NaN,NaN,HOME,N,1,$0,$0,$0,$0,$0,$200,200.0,NaN,Y
3,4,47141.0,31,NaN,Female,N,Y,N,Y,NaN,NaN,HOME,Y,0,$0,$0,$0,$0,$0,$0,0.0,12/3/1987,N
4,5,92555.0,68,NaN,Female,N,N,N,N,NaN,NaN,HOME,Y,0,$0,$0,$0,$0,$0,$0,505.0,9/11/1950,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34503,34504,7848.0,42,NaN,Female,N,N,N,N,NaN,NaN,HOME,N,0,$0,$0,$0,$0,$0,$0,0.0,NaN,N
34504,34505,28275.0,24,NaN,Male,N,N,N,N,"$250,000-$499,999",NaN,CAMP,Y,0,$0,$0,$0,$0,$0,$0,80.0,9/23/1994,Y
34505,34506,42539.0,27,NaN,Female,N,Y,N,Y,NaN,UB,HOME,Y,0,$0,$0,$0,$0,$0,$0,0.0,1/3/1991,N
34506,34507,32733.0,46,Married,Female,N,N,N,Y,NaN,NaN,HOME,Y,1,$0,$0,$0,$120,$0,$0,120.0,5/11/1972,Y


In [12]:
# Missing Value Standardization Result
df.isnull().sum().sort_values(ascending=False)

WealthRating                 31799
AcademicDegreeLevel          26902
MaritalStatus                24568
DonorDateOfBirth             21190
PreferredAddressType          4043
GenderIdentity                 493
DonorPostalCode                 91
LastFiscalYearDonation           0
CumulativeDonationAmount         0
CurrentFiscalYearDonation        0
Donation5FiscalYearsAgo          0
Donation4FiscalYearsAgo          0
Donation3FiscalYearsAgo          0
Donation2FiscalYearsAgo          0
DonorUniqueId                    0
ConsecutiveDonorYears            0
HasEmailFlag                     0
HasInvolvementFlag               0
IsParentFlag                     0
IsAlumnusFlag                    0
IsMemberFlag                     0
DonorAge                         0
DonorIndicatorFlag.              0
dtype: int64

## Missing Value Standardization Result
The dataset was inspected for common placeholder values including:
- NA
- N/A
- NULL
- null
- None
- none
- ?
- blank strings
- whitespace-only strings

After applying missing-value standardization procedures, missing-value counts remained unchanged across all columns. This indicates that the dataset already primarily uses proper NaN values to represent missing information.

In [13]:
# Check Duplicate Rows
dupe_rows = df.duplicated().sum()
print(f"Duplicate Rows: {dupe_rows}")

# Check Duplicate Donor IDs
dupe_ids = df["DonorUniqueId"].duplicated().sum()
print(f"Duplicate Donor IDs: {dupe_ids}")

# Confirm all Donor IDs are unique
print("Total Donors: ", len(df))
print("Unique Donors:", df["DonorUniqueId"].nunique())

Duplicate Rows: 0
Duplicate Donor IDs: 0
Total Donors:  34508
Unique Donors: 34508


## Duplicate Analysis

Both records and donor identifiers were checked for duplicates in the dataset.

## Exact Duplicate Records
- No rows were exact duplicates in the dataset
- This suggests that each record contains a unqiue combination of feature values

## Donor Identifier Validation
- No duplicate values were identified in the DonorUniqueId field
- This is further proven as the number of unique donor identifiers matches the total number of dataset records
- This is important to check as the DonorUniqueId is the primary key for the dataset and all values must be unique

## Conclusion
- No duplicate records were identified
- No records had to be removed during this step of preprocessing
- The dataset appears to be aggregated at the donor level, with one record per donor

In [14]:
# Rename column names from Pascal Case to Snake Case
df.columns = [
    re.sub(r"(?<!^)(?=[A-Z])|(?<![0-9])(?=[0-9])", "_", col.strip().replace(".", "")).lower()
    for col in df.columns
]

for col in df.columns:
    print(f"'{col}',")

'donor_unique_id',
'donor_postal_code',
'donor_age',
'marital_status',
'gender_identity',
'is_member_flag',
'is_alumnus_flag',
'is_parent_flag',
'has_involvement_flag',
'wealth_rating',
'academic_degree_level',
'preferred_address_type',
'has_email_flag',
'consecutive_donor_years',
'last_fiscal_year_donation',
'donation_2_fiscal_years_ago',
'donation_3_fiscal_years_ago',
'donation_4_fiscal_years_ago',
'donation_5_fiscal_years_ago',
'current_fiscal_year_donation',
'cumulative_donation_amount',
'donor_date_of_birth',
'donor_indicator_flag',


## Column Name Standardization

All column names were successfully standardized from Pascal Case to Snake Case formatting.

The following conventions were applied:
- Converted all column names to lowercase
- Inserted underscores between words
- Inserted underscores between numbers without splitting up multi-digit numbers
- Removed special characters and punctuation

Examples:
- DonorUniqueId -> donor_unique_id
- CurrentFiscalYearDonation -> current_fiscal_year_donation
- DonorIndicatorFlag. -> donor_indicator_flag

In [15]:
# Donation columns datatype
donation_cols = [
    "last_fiscal_year_donation",
    "donation_2_fiscal_years_ago",
    "donation_3_fiscal_years_ago",
    "donation_4_fiscal_years_ago",
    "donation_5_fiscal_years_ago",
    "current_fiscal_year_donation",
]

df[donation_cols].dtypes

last_fiscal_year_donation       str
donation_2_fiscal_years_ago     str
donation_3_fiscal_years_ago     str
donation_4_fiscal_years_ago     str
donation_5_fiscal_years_ago     str
current_fiscal_year_donation    str
dtype: object

In [16]:
# Convert donation columns datatype from string to float
for col in donation_cols:
    df[col] = (
        df[col].replace(r"[\$,]", "", regex=True).astype(float)
    )

df[donation_cols].dtypes

last_fiscal_year_donation       float64
donation_2_fiscal_years_ago     float64
donation_3_fiscal_years_ago     float64
donation_4_fiscal_years_ago     float64
donation_5_fiscal_years_ago     float64
current_fiscal_year_donation    float64
dtype: object

In [17]:
# Date Column datatype
df["donor_date_of_birth"].info()

<class 'pandas.Series'>
RangeIndex: 34508 entries, 0 to 34507
Series name: donor_date_of_birth
Non-Null Count  Dtype
--------------  -----
13318 non-null  str  
dtypes: str(1)
memory usage: 389.7 KB


In [18]:
# Convert Date column from string to datetime
df["donor_date_of_birth"] = (
    pd.to_datetime(df["donor_date_of_birth"], errors="coerce")
    .astype("datetime64[ns]")
)

df["donor_date_of_birth"].info()

<class 'pandas.Series'>
RangeIndex: 34508 entries, 0 to 34507
Series name: donor_date_of_birth
Non-Null Count  Dtype         
--------------  -----         
13318 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 269.7 KB


In [19]:
# Binary Columns datatype

binary_cols = [
    "is_member_flag",
    "is_alumnus_flag",
    "is_parent_flag",
    "has_involvement_flag",
    "has_email_flag",
    "donor_indicator_flag",
]

df[binary_cols].dtypes

is_member_flag          str
is_alumnus_flag         str
is_parent_flag          str
has_involvement_flag    str
has_email_flag          str
donor_indicator_flag    str
dtype: object

In [20]:
df[binary_cols].head()

,is_member_flag,is_alumnus_flag,is_parent_flag,has_involvement_flag,has_email_flag,donor_indicator_flag
0,N,N,N,N,N,Y
1,N,Y,N,Y,Y,Y
2,N,N,N,N,N,Y
3,N,Y,N,Y,Y,N
4,N,N,N,N,Y,Y


In [21]:
# Convert Binary Columns from string to boolean values (0 or 1)
binary_map = {
    "N": 0,
    "Y": 1
}

for col in binary_cols:
    if df[col].dtype in ['object', 'string']:
        df[col] = df[col].map(binary_map).astype("Int64")

df[binary_cols].dtypes

is_member_flag          Int64
is_alumnus_flag         Int64
is_parent_flag          Int64
has_involvement_flag    Int64
has_email_flag          Int64
donor_indicator_flag    Int64
dtype: object

In [22]:
df[binary_cols].head()

,is_member_flag,is_alumnus_flag,is_parent_flag,has_involvement_flag,has_email_flag,donor_indicator_flag
0,0,0,0,0,0,1
1,0,1,0,1,1,1
2,0,0,0,0,0,1
3,0,1,0,1,1,0
4,0,0,0,0,1,1


In [23]:
df.dtypes

donor_unique_id                          int64
donor_postal_code                      float64
donor_age                                int64
marital_status                             str
gender_identity                            str
is_member_flag                           Int64
is_alumnus_flag                          Int64
is_parent_flag                           Int64
has_involvement_flag                     Int64
wealth_rating                              str
academic_degree_level                      str
preferred_address_type                     str
has_email_flag                           Int64
consecutive_donor_years                  int64
last_fiscal_year_donation              float64
donation_2_fiscal_years_ago            float64
donation_3_fiscal_years_ago            float64
donation_4_fiscal_years_ago            float64
donation_5_fiscal_years_ago            float64
current_fiscal_year_donation           float64
cumulative_donation_amount             float64
donor_date_of

In [24]:
for col in [
    "last_fiscal_year_donation",
    "donation_2_fiscal_years_ago",
    "donation_3_fiscal_years_ago",
    "donation_4_fiscal_years_ago",
    "donation_5_fiscal_years_ago",
    "current_fiscal_year_donation",
]:
    print(f"\n{col}")
    print(df[col].sample(10, random_state=RANDOM_STATE).tolist())

print("\nDate Column Sample")
print(df["donor_date_of_birth"].sample(10, random_state=RANDOM_STATE))

print("\nBinary Column Samples")
binary_cols = [
    "is_member_flag",
    "is_alumnus_flag",
    "is_parent_flag",
    "has_involvement_flag",
    "has_email_flag",
    "donor_indicator_flag"
]

for col in binary_cols:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))


last_fiscal_year_donation
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 25.0, 0.0]

donation_2_fiscal_years_ago
[0.0, 0.0, 0.0, 0.0, 100.0, 0.0, 0.0, 0.0, 0.0, 0.0]

donation_3_fiscal_years_ago
[0.0, 0.0, 0.0, 0.0, 100.0, 0.0, 0.0, 0.0, 0.0, 0.0]

donation_4_fiscal_years_ago
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

donation_5_fiscal_years_ago
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 150.0, 0.0, 0.0, 0.0]

current_fiscal_year_donation
[0.0, 0.0, 0.0, 0.0, 0.0, 200.0, 0.0, 0.0, 0.0, 0.0]

Date Column Sample
16199   1973-08-16
25925   1992-01-03
12681   1972-03-19
34020          NaT
32576   1958-12-20
9047           NaT
2988           NaT
31189   1954-06-15
2661    1962-03-17
20668   1990-09-20
Name: donor_date_of_birth, dtype: datetime64[ns]

Binary Column Samples

is_member_flag
is_member_flag
0    34508
Name: count, dtype: Int64

is_alumnus_flag
is_alumnus_flag
0    26086
1     8422
Name: count, dtype: Int64

is_parent_flag
is_parent_flag
0    31807
1     2701
Name: count, dtype: Int64

h

## Datatype Conversion

- All donation related fields were converted from string representations containing currency symbols into numeric floating point values
- The donor_date_of_birth field was converted from string into a datetime datatype
- Binary indicator columns were converted from Y/N to numerical encoding
  - Y -> 1
  - N -> 0

In [25]:
df[donation_cols].describe()

,last_fiscal_year_donation,donation_2_fiscal_years_ago,donation_3_fiscal_years_ago,donation_4_fiscal_years_ago,donation_5_fiscal_years_ago,current_fiscal_year_donation
count,3.450800e+04,3.450800e+04,34508.000000,34508.000000,3.450800e+04,3.450800e+04
mean,3.776168e+02,9.620105e+01,63.754926,57.118813,1.266312e+02,1.977946e+02
std,5.404837e+04,1.045471e+04,3282.195711,2594.556590,1.006439e+04,1.585057e+04
min,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00
75%,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00
max,1.000000e+07,1.932960e+06,500000.000000,250000.000000,1.250000e+06,2.000000e+06


In [26]:
# Check for negative donations
for col in donation_cols:
    negatives = (df[col] < 0).sum()
    print(f"{col}: {negatives}")

last_fiscal_year_donation: 0
donation_2_fiscal_years_ago: 0
donation_3_fiscal_years_ago: 0
donation_4_fiscal_years_ago: 0
donation_5_fiscal_years_ago: 0
current_fiscal_year_donation: 0


In [27]:
# Check maximum values
df[donation_cols].max().sort_values(ascending=False)

last_fiscal_year_donation       10000000.0
current_fiscal_year_donation     2000000.0
donation_2_fiscal_years_ago      1932960.0
donation_5_fiscal_years_ago      1250000.0
donation_3_fiscal_years_ago       500000.0
donation_4_fiscal_years_ago       250000.0
dtype: float64

In [28]:
# Donations columns datatypes
df[donation_cols].dtypes

last_fiscal_year_donation       float64
donation_2_fiscal_years_ago     float64
donation_3_fiscal_years_ago     float64
donation_4_fiscal_years_ago     float64
donation_5_fiscal_years_ago     float64
current_fiscal_year_donation    float64
dtype: object

In [29]:
# Look at top 10 largest donations
for col in donation_cols:
    print(f"\n{col}")
    print(df[col].nlargest(10))


last_fiscal_year_donation
18404    10000000.0
1324       500000.0
2172       500000.0
34056      500000.0
32411      100750.0
27583       99954.0
6708        90000.0
27237       90000.0
33879       78419.0
23329       52325.0
Name: last_fiscal_year_donation, dtype: float64

donation_2_fiscal_years_ago
1324     1932960.0
27237     100000.0
33879     100000.0
3286       53000.0
33680      45100.0
32924      30000.0
10571      28560.0
20685      27000.0
6708       25750.0
430        25000.0
Name: donation_2_fiscal_years_ago, dtype: float64

donation_3_fiscal_years_ago
2172     500000.0
18404    199842.0
34056    199842.0
23329     97850.0
27237     97850.0
6708      75000.0
20878     75000.0
23313     56500.0
30244     50000.0
27583     34000.0
Name: donation_3_fiscal_years_ago, dtype: float64

donation_4_fiscal_years_ago
9474     250000.0
33186    250000.0
2928     180100.0
33766    180100.0
27075    106000.0
29205    100000.0
23313     54000.0
14343     53505.0
22418     53505.0
3286  

In [30]:
# Check for missing values
df[donation_cols].isnull().sum()

last_fiscal_year_donation       0
donation_2_fiscal_years_ago     0
donation_3_fiscal_years_ago     0
donation_4_fiscal_years_ago     0
donation_5_fiscal_years_ago     0
current_fiscal_year_donation    0
dtype: int64

## Donation Value Validation
- Several donor identifiers appeared repeatedly among the highest donation amounts across multiple fiscal years
  - This suggests the largest values are associated with a small number of major donors rather than it being an mistake with the dataset
  - Most large donations appear consistent with historical donation behavior
- One donor recorded a donation of $10,000,000 during the last fiscal year, which is substantially larger than other donations observed in the dataset.
  - While this value may represent a legitimate major donation, it will be flagged for additional review during outlier analysis

In [31]:
# Check date range of date of birth
print("Min DOB:", df["donor_date_of_birth"].min())
print("Max DOB:", df["donor_date_of_birth"].max())

Min DOB: 1908-09-08 00:00:00
Max DOB: 2017-09-01 00:00:00


In [32]:
# Check if there is a date of birth that is in the future
future_birthdays = df[
    df["donor_date_of_birth"] > pd.Timestamp.today()
]

print("Future Birthdays:", len(future_birthdays))

Future Birthdays: 0


In [33]:
# Check for donors that are older than 110
older_than_110 = df[df["donor_age"] > 110]

print("Donors Older Than 110:", len(older_than_110))

Donors Older Than 110: 0


In [34]:
df["donor_age"].describe()

count    34508.000000
mean        43.366987
std         11.405722
min          1.000000
25%         42.000000
50%         42.000000
75%         42.000000
max        110.000000
Name: donor_age, dtype: float64

In [35]:
df[df["donor_age"] < 18]["donor_age"].value_counts().sort_index()

donor_age
1      1
3     12
4      9
5     14
6      6
7     13
8      5
9      5
10     6
11    10
12     4
13     6
14     7
15     7
16    23
17    29
Name: count, dtype: int64

In [36]:
# Calculate age from date of birth
CURRENT_YEAR = pd.Timestamp.today().year

df["calculated_age"] = (
    CURRENT_YEAR -
    df["donor_date_of_birth"].dt.year
)

# Compare provided age vs calculated age
df["age_difference"] = (
    df["donor_age"] -
    df["calculated_age"]
)

In [37]:
# Check for age differences due to dataset date offset
print("Age Difference Distribution")
print(df["age_difference"].value_counts(dropna=False).sort_index())

print("\nSummary Statistics")
print(df["age_difference"].describe())

print("\nPotential Age Mismatches (> 1 year difference)")
display(
    df.loc[
        df["age_difference"].abs() > 1,
        [
            "donor_unique_id",
            "donor_age",
            "donor_date_of_birth",
            "calculated_age",
            "age_difference"
        ]
    ].head(20)
)

Age Difference Distribution
age_difference
-8.0    13318
 NaN    21190
Name: count, dtype: int64

Summary Statistics
count    13318.0
mean        -8.0
std          0.0
min         -8.0
25%         -8.0
50%         -8.0
75%         -8.0
max         -8.0
Name: age_difference, dtype: float64

Potential Age Mismatches (> 1 year difference)


,donor_unique_id,donor_age,donor_date_of_birth,calculated_age,age_difference
1,2,33,1985-06-16,41.0,-8.0
3,4,31,1987-12-03,39.0,-8.0
4,5,68,1950-09-11,76.0,-8.0
5,6,57,1961-01-23,65.0,-8.0
10,11,40,1978-09-14,48.0,-8.0
15,16,47,1971-12-15,55.0,-8.0
22,23,61,1957-09-08,69.0,-8.0
23,24,58,1960-08-09,66.0,-8.0
29,30,62,1956-01-03,70.0,-8.0
30,31,63,1955-05-11,71.0,-8.0


In [38]:
# Check sample of donor age and date of birth
sample = df[
    df["donor_date_of_birth"].notna()
][[
    "donor_age",
    "donor_date_of_birth"
]].head(10)

sample

,donor_age,donor_date_of_birth
1,33,1985-06-16
3,31,1987-12-03
4,68,1950-09-11
5,57,1961-01-23
10,40,1978-09-14
15,47,1971-12-15
22,61,1957-09-08
23,58,1960-08-09
29,62,1956-01-03
30,63,1955-05-11


In [39]:
print("Future DOBs:",
      (df["donor_date_of_birth"] > pd.Timestamp.today()).sum())

print("Age > 110:",
      (df["donor_age"] > 110).sum())

print("Age < 18:",
      (df["donor_age"] < 18).sum())

Future DOBs: 0
Age > 110: 0
Age < 18: 157


In [40]:
# Check for donors that are younger than 18
under_18 = df[df["donor_age"] < 18]

print("Donors Younger Than 18:", len(under_18))
under_18[[
    "donor_unique_id",
    "donor_age",
    "donor_date_of_birth"
]].head()

Donors Younger Than 18: 157


,donor_unique_id,donor_age,donor_date_of_birth
171,172,12,2006-12-15
234,235,3,2015-05-16
272,273,7,2011-12-01
498,499,7,2011-05-11
737,738,7,2011-01-17


In [41]:
# Check underage donors
underage = df[df["donor_age"] < 18]

print("Underage Records:", len(underage))

display(
    underage[
        [
            "donor_age",
            "donor_indicator_flag",
            "is_parent_flag",
            "is_alumnus_flag",
            "has_involvement_flag",
            "current_fiscal_year_donation",
            "cumulative_donation_amount"
        ]
    ].head(20)
)

Underage Records: 157


,donor_age,donor_indicator_flag,is_parent_flag,is_alumnus_flag,has_involvement_flag,current_fiscal_year_donation,cumulative_donation_amount
171,12,1,0,0,0,5.0,5.0
234,3,0,0,0,0,0.0,0.0
272,7,0,0,0,0,0.0,0.0
498,7,1,0,0,0,3000.0,3325.0
737,7,0,0,0,0,0.0,0.0
833,7,0,0,0,0,0.0,0.0
977,17,0,0,0,0,0.0,0.0
1042,4,1,0,0,0,0.0,20.0
1050,3,1,1,1,0,0.0,84.0
1832,4,1,0,0,0,0.0,40.0


In [42]:
df.groupby(df["donor_age"] < 18)["cumulative_donation_amount"].describe()

,count,mean,std,min,25%,50%,75%,max
donor_age,,,,,,,,
False,34351.0,2372.942796,114061.067804,0.0,0.0,25.0,144.0,12221854.0
True,157.0,304.968153,1744.146013,0.0,0.0,5.0,115.0,21120.0


# Date Cleaning & Temporal Validation

## Date Conversion
- The donor_date_of_birth column was converted from a string format to a datetime datatype
- Missing birth dates are now represented as NaT

### Age and Date of Birth Consistency
- The provided donor_age field was compared against ages calculated directly from donor_date_of_birth
- Calculated ages are consistently 8 years older than the provided donor_age field across all valid records, suggesting that the donor_age field was calculated at a fixed point in time and was not subsequently updated.
- The age discrepancy appears to be a dataset timing artifact rather than a data quality issue

## Temporal Validation
- Validation Checks & Results:
  - Future Birth Dates: 0
  - Donors older than 110: 0
  - Donors younger than 18: 157

### Future Birth Dates
- No donor records contained birth dates occurring in the future, indicating that the date field does not contain obvious temporal inconsistencies

### Extreme Ages
- No donors were identified with ages greater than 110 years old, suggesting that the age field remains within a reasonable human lifespan range

### Underage
- The underage donor population spans ages 1 through 17
- Notably, several records are associated with very young ages, including ages 1 through 10
  - These values are unusual within a donor dataset because individuals of these ages are generally not expected to make independent donation decisions
- Potential Explanations:
  - Household-linked records
  - Family donor records
  - Dependent records
  - Administrative records associated with donor households

## Note: 
For future modeling, donor_age will likely be preferred over donor_date_of_birth because:
- Age is already available as a feature
- donor_age contains no missing values, while donor_date_of_birth contains 21,190 missing values (61.4% of records)
- Age is generally more interpretable for modeling purposes
- Exact birth dates introduce unnecessary granularity

In [43]:
# Inspect unique values
categorical_cols = [
    "marital_status",
    "gender_identity",
    "wealth_rating",
    "academic_degree_level",
    "preferred_address_type"
]

for col in categorical_cols:
    print(f"\n{'='*60}")
    print(col)
    print(f"{'='*60}")
    print(sorted(df[col].dropna().unique()))


marital_status
['Divorced', 'Married', 'Never Married', 'Separated', 'Single', 'Widowed']

gender_identity
['Female', 'Male', 'U', 'Uknown', 'Unknown']

wealth_rating
['$1,000,000-$2,499,999', '$1-$24,999', '$100,000-$249,999', '$2,500,000-$4,999,999', '$25,000-$49,999', '$250,000-$499,999', '$50,000-$99,999', '$500,000-$999,999']

academic_degree_level
['GC', 'GD', 'GM', 'GP', 'UB', 'UC', 'UG']

preferred_address_type
['BUSN', 'CAMP', 'HOME', 'OTR']


In [44]:
# Merge Never Married and Single together to just Single
df["marital_status"] = df["marital_status"].replace({
    "Never Married": "Single"
})

df["marital_status"].value_counts(dropna=False)

marital_status
NaN          24568
Married       6547
Single        3143
Widowed        157
Divorced        84
Separated        9
Name: count, dtype: int64

In [45]:
# Change U & Uknown to Unknown in gender_identity
df["gender_identity"] = df["gender_identity"].replace({
    "Uknown": "Unknown",
    "U": "Unknown"
})

df["gender_identity"].value_counts(dropna=False)

gender_identity
Female     16678
Male       16233
Unknown     1104
NaN          493
Name: count, dtype: int64

In [46]:
# Expand the names of preferred_address_type
df["preferred_address_type"] = (
    df["preferred_address_type"]
    .replace({
        "HOME": "Home",
        "BUSN": "Business",
        "CAMP": "Campus",
        "OTR": "Other"
    })
)

df["preferred_address_type"].value_counts(dropna=False)

preferred_address_type
Home        28771
NaN          4043
Business      976
Campus        647
Other          71
Name: count, dtype: int64

In [47]:
# Inspect unique values
for col in categorical_cols:
    print(f"\n{'='*60}")
    print(col)
    print(f"{'='*60}")
    print(df[col].value_counts(dropna=False))


marital_status
marital_status
NaN          24568
Married       6547
Single        3143
Widowed        157
Divorced        84
Separated        9
Name: count, dtype: int64

gender_identity
gender_identity
Female     16678
Male       16233
Unknown     1104
NaN          493
Name: count, dtype: int64

wealth_rating
wealth_rating
NaN                      31799
$50,000-$99,999            645
$1-$24,999                 580
$25,000-$49,999            564
$100,000-$249,999          511
$250,000-$499,999          265
$500,000-$999,999           81
$1,000,000-$2,499,999       59
$2,500,000-$4,999,999        4
Name: count, dtype: int64

academic_degree_level
academic_degree_level
NaN    26902
UB      3648
GM      3112
GP       571
GD       173
GC        74
UC        19
UG         9
Name: count, dtype: int64

preferred_address_type
preferred_address_type
Home        28771
NaN          4043
Business      976
Campus        647
Other          71
Name: count, dtype: int64


## Categorical Value Standardization

### Marital Status
- The following values were identified:
  - Married
  - Single
  - Widowed
  - Divorced
  - Separated
  - Never Married
- the category Never Married was standardized to Single because both categories represent individuals who are not currently married

### Gender
- The following values were identified:
  - Female
  - Male
  - Uknown
  - Unknown
  - U
- The categories Uknown and U were standardized to Unknown reducing duplicate representations of the same category

### Wealth Rating
- No modifications were required

### Academic Degree Level
- AcademicDegreeLevel contains coded categorical values whose meanings could not be validated using the available dataset documentation
- To avoid introducing incorrect assumptions, no modifications were made to this field during preprocessing

### Preferred Address Type
- The following values were identified:
  - HOME
  - BUSN
  - CAMP
  - OTR
- These were expanded to imporve readability and interpretability
  - HOME -> Home
  - BUSN -> Business
  - CAMP -> Campus
  - OTR -> Other

In [48]:
# Compare the total donations from the past 5 years with their cumulative donations
df["five_year_total"] = (
    df["last_fiscal_year_donation"]
    + df["donation_2_fiscal_years_ago"]
    + df["donation_3_fiscal_years_ago"]
    + df["donation_4_fiscal_years_ago"]
    + df["donation_5_fiscal_years_ago"]
    + df["current_fiscal_year_donation"]
)

suspicious_donations = df[
    df["cumulative_donation_amount"] <
    df["five_year_total"]
]

print("Number of records where 5-year totals exceed lifetime cumulative amount:", len(suspicious_donations))

Number of records where 5-year totals exceed lifetime cumulative amount: 3


In [49]:
# Display the donations where the 5-year totals exceed lifetime cumulative amount
display(
    suspicious_donations[
        [
            "donor_unique_id",
            "five_year_total",
            "cumulative_donation_amount"
        ]
    ]
)

,donor_unique_id,five_year_total,cumulative_donation_amount
1869,1870,3327.0,3326.0
5754,5755,13436.0,13435.0
14559,14560,497.0,496.0


In [50]:
# 
print(f"Percentage of records where the 5-year totals exceed lifetime cumulative amount: {len(suspicious_donations) / len(df) * 100:.6f}%")

Percentage of records where the 5-year totals exceed lifetime cumulative amount: 0.008694%


## Cumulative Donation Validation

A data integrity check revealed that 3 records contained a minor discrepancy
- Discrepeancy: Their calculated 5-year donation total exceeded their lifetime cumulative amount.
- Further analysis showed that in all three cases, the variance between the two is exactly $1
- This $1 difference is negligible and is likely due to adjustments or rounding erros rather than data-entry errors.
- Due to the differences being negligible, no changes were made  

In [51]:
# Summary donation statistics 
donation_cols = [
    "last_fiscal_year_donation",
    "donation_2_fiscal_years_ago",
    "donation_3_fiscal_years_ago",
    "donation_4_fiscal_years_ago",
    "donation_5_fiscal_years_ago",
    "current_fiscal_year_donation",
    "cumulative_donation_amount"
]

df[donation_cols].describe()

,last_fiscal_year_donation,donation_2_fiscal_years_ago,donation_3_fiscal_years_ago,donation_4_fiscal_years_ago,donation_5_fiscal_years_ago,current_fiscal_year_donation,cumulative_donation_amount
count,3.450800e+04,3.450800e+04,34508.000000,34508.000000,3.450800e+04,3.450800e+04,3.450800e+04
mean,3.776168e+02,9.620105e+01,63.754926,57.118813,1.266312e+02,1.977946e+02,2.363534e+03
std,5.404837e+04,1.045471e+04,3282.195711,2594.556590,1.006439e+04,1.585057e+04,1.138014e+05
min,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00,2.500000e+01
75%,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000e+00,1.440000e+02
max,1.000000e+07,1.932960e+06,500000.000000,250000.000000,1.250000e+06,2.000000e+06,1.222185e+07


In [52]:
# Check quantiles
for col in donation_cols:
    print(f"\n{col}")
    print(df[col].quantile([
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.999
    ]))


last_fiscal_year_donation
0.500       0.00
0.750       0.00
0.900       0.00
0.950      20.00
0.990     262.79
0.999    7504.93
Name: last_fiscal_year_donation, dtype: float64

donation_2_fiscal_years_ago
0.500       0.000
0.750       0.000
0.900       0.000
0.950      25.000
0.990     349.930
0.999    6058.667
Name: donation_2_fiscal_years_ago, dtype: float64

donation_3_fiscal_years_ago
0.500       0.0
0.750       0.0
0.900       0.0
0.950      25.0
0.990     276.0
0.999    6325.0
Name: donation_3_fiscal_years_ago, dtype: float64

donation_4_fiscal_years_ago
0.500       0.00
0.750       0.00
0.900       0.00
0.950      24.00
0.990     250.00
0.999    3187.34
Name: donation_4_fiscal_years_ago, dtype: float64

donation_5_fiscal_years_ago
0.500       0.0
0.750       0.0
0.900       0.0
0.950      11.0
0.990     200.0
0.999    4849.3
Name: donation_5_fiscal_years_ago, dtype: float64

current_fiscal_year_donation
0.500       0.00
0.750       0.00
0.900       0.00
0.950       1.00
0.990  

## Donation Outlier Analysis

For most annual donation fields: 
- Median Donation Amount: $0

- 75th Percentile: $0

- 90th Percentile: $0

Meaningful donation amounts do not begin appearing until approximately the 95th percentile, indicating that the majority of donors do not contribute during a given fiscal year while a small number of donors account for a disproportionately large share of donations

The distribution in cumulative_donation_amount is much healthier than the annual donation fields: 
- Median = $25
  
- 75th percentile = $144

- 90th percentile = $525

- 95th percentile = $1,309

- 99th percentile = $9,599

This suggests cumulative_donation_amount is a stronger predictor in the dataset because it captures long term donor behavior rather than a single year's activity.



Several donation fields contain extremely large values, including donations exceeding:
- $500,000

- $1,000,000

- $10,000,000 

These records were previously investigated and appear to represent legitimate major donors rather than data-entry errors. Donation-related variables exhibit extreme positive skew but do not contain evidence of invalid outlier values.

No outlier removal, winsorization, or value capping was performed during this analysis. Because the objective of this project is donor prediction, high value donors represent important business observations and should be preserved. 
